本笔记针对的乃是纯小白，大一刚学完C语言,然后稍微了解了一点python的语法

首先是最简单的MLP
然后是LN，residual，
然后是尝试自注意力
最后

In [9]:
from dataclasses import dataclass

@dataclass
class Config:
    block_size: int=1024#
    vocab_size: int=50304
    n_layer: int= 12
    n_head: int=12
    n_embd: int=768
    dropout: float=0.1
    bias: bool = False #本月听到的最大的笑话：发现有时候不加偏置训练更稳定，结果也不会出问题
    

#关于这个dataclasses这东西，上面写法是等价于下面这个的
# class Config:
#     def __init__(self,
#              block_size=1024,
#              vocab_size=50304,
#              n_layer=12,
#              n_head=12,
#              n_embd=768,
#              dropout=0.1,
#              bias=False
#              ):
#         self.block_size = block_size
#         self.vocab_size = vocab_size
#         self.n_layer = n_layer
#         self.n_head = n_head
#         self.n_embd = n_embd
#         self.bias = bias

    
config=Config()

#配置项，超参数自行设置








In [10]:
import torch
import torch.nn as nn



class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        #扩embedding维度
        self.c_fc =nn.Linear(config.n_embd, 4*config.n_embd,bias = config.bias )
        self.gelu=nn.GELU()
        #再回去
        self.c_proj=nn.Linear(4*config.n_embd,config.n_embd,bias=config.bias)
        self.dropout=nn.Dropout(config.dropout)
#__init__()里是所有可能用到的函数，真正怎么用的是看forward函数

    def forward(self,x):
        x=self.c_fc(x)
        x=self.gelu(x)
        x=self.c_proj(x)
        x=self.dropout(x)
        return x
    
#最原始的MLP，不赘述

#PS: 这里，额没啥



In [11]:
#写这个基本原理清楚的话，后面只需要维度对的上就没有问题了
#是整个架构最关键的难点
import math
import torch.nn.functional as F

class CausalAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_atten = nn.Linear(config.n_embd, 3*config.embd,bias=config.bias)
        #这个是用来计算QKV的

    def forward(self,x):
        B,T,C = x.size
        #这个得到x的三个维度，B是批次，或者说样本个数，T指的是一句话多少个token，然后C就是n_embd
        
        q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        k=k.view(B,T,config.n_head,C//config.n_head).transpose(1,2)
        q=q.view(B,T,config.n_head,C//config.n_head).transpose(1,2)
        v=v.view(B,T,config.n_head,C//config.n_head).transpose(1,2)
        #这个就是计算QKV然后交换一下，把n_head放到前面，方便后面点积矩阵是默认最后两个维度表示矩阵的

        atten=(q@k.transpose(-2.-1))*(1.0/math.sqrt(q.size(-1)))
        #这里有个特别简单的记法，就是我们需要计算的是每个token对其他token的注意力是吧
        #这意味着我们最后得到的矩阵应该是T*T的size，所以就很明显了啊，把transpose我们的k
        #最后归一化，因为和的维度是最后一位，所以除个最后一位


        atten=atten.masked_fill((self.bias[:,:,:T:T])==0,float('-inf'))
        #这个是掩码的关键，就是这个self.bias是一个梯形的，可以广播成和atten相同的size，然后就满足条件(self.bias[:,:,:T:T])==0的位置会被赋值float('-inf')
        atten=F.softmax(atten,dim=-1)
        atten=self.attn_dropout(atten)


        y=atten@v 
        y=y.transppose(1,2).contigous().view(B,T,C)#.contigous是处理内存用的，因为这个刚转置后的内存是乱的，所以需要如此处理一下然后再.view
        return self.resid_dropout(self.c_proj(y))
    
        #就是过程中如果有步理解的可以直接把维度写出来，记录每一步的维度变化，会好很多



In [12]:
class Block(nn.Module):

    def __init__(self,config):
        self.ln1=LayerNorm(config.n_embd,bias=config.bias)
        self.attn = CausalAttention(config)
        self.ln2=LayerNorm(config.n_embd,bias=config.bias)
        self.mlp=MLP(config)

    def forward(self,x):
        x=x+self.atten(self.ln1(x))
        x=x+self.mlp(self.ln2(x))
        return x
    
#这个就是最后的block，基础架构    